# Teste isolado — ARPE (Agência de Regulação de Pernambuco)

Fonte candidata: **ARPE** — já existe linha placeholder em `controle_fontes`
(`nome_fonte='ARPE'`, `setor='Saneamento'`, `tipo_categoria='Agências
Reguladoras Estaduais (PE)'`, `importancia_original='Alta'`, `source_id='—'`).
Notebook **descartável** (Fase 1) — sem dispatcher, sem `atualizar_status_fonte`,
sem gravar nada em produção.

## Confirmado antes de assumir (testado contra o site real, não HTML salvo)

- **Site**: Joomla 5 (`arpe.pe.gov.br`), sem proteção anti-bot visível — uma
  requisição HTTP simples com `User-Agent` de navegador comum recebeu HTTP 200
  normalmente.
- **`robots.txt`**: não bloqueia `/legislacao/` (só bloqueia caminhos internos
  do Joomla: `/administrator/`, `/cache/`, etc.).
- **Não é um site de notícias** — não há RSS nem seção de "notícias" separada.
  O conteúdo relevante são atos normativos (Resoluções, Portarias, Deliberações
  do Conselho Consultivo). Este teste cobre **Resoluções Arpe**
  (`/legislacao/resolucoes-arpe`) — o tipo de ato mais central, análogo ao que
  já foi feito para Agesan-RS.
- **Paginação**: **nenhuma** — é uma única página listando o histórico
  completo, da Resolução Nº 001/2001 até a mais recente. Isso significa que a
  primeira execução em produção vai baixar ~300 PDFs de uma vez (backfill
  histórico); dedup por manifesto evita reprocessamento nas execuções
  seguintes.
- **Posição da data**: embutida no próprio texto da listagem (formato
  `"RESOLUÇÃO ARPE Nº 328, DE 24 DE FEVEREIRO DE 2026"`), junto com a
  descrição — não precisa abrir página individual para a maioria dos itens.
- **Formato do site — particularidade real encontrada**: cada item da
  listagem é um bloco com (a) um `<div>` de texto solto contendo
  `"RESOLUÇÃO ARPE Nº X, DE ... DE ..."` + descrição, seguido de (b) um
  `<a href="...pdf">` **separado**, cujo texto do link é só
  `"Resolução Arpe Nº X/AAAA"` (sem data). Ou seja, **o título/data não estão
  no próprio link** — é preciso pareamento por ordem de aparição no HTML
  (título mais recente visto "na frente" de cada link), não dá pra usar o
  `extrair_links_pdf()` genérico do dispatcher (que só lê o texto do `<a>`).
- **~25 resoluções (das mais recentes de 2026, e algumas antigas esparsas) não
  têm link `.pdf` direto** — em vez disso, o link vai para uma página HTML
  própria (`/resolucao-arpe-n-345`) com o texto integral da resolução
  embutido, sem nenhum PDF associado. Isso não se encaixa no contrato do
  `processar_pdf()` do dispatcher genérico (que baixa bytes e extrai via
  `pypdf`) — fica de fora do escopo desta primeira integração, documentado
  como pendência conhecida (mesmo padrão da pendência já registrada para as
  variações da Agesan-RS em `CLAUDE.md`).

In [ ]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import re
import urllib.parse

import httpx
from bs4 import BeautifulSoup, NavigableString

In [ ]:
URL_RESOLUCOES = "https://www.arpe.pe.gov.br/legislacao/resolucoes-arpe"

MESES_PT = {
    "janeiro": 1, "fevereiro": 2, "março": 3, "marco": 3, "abril": 4,
    "maio": 5, "junho": 6, "julho": 7, "agosto": 8, "setembro": 9,
    "outubro": 10, "novembro": 11, "dezembro": 12,
}

TITULO_RE = re.compile(
    r"RESOLU[ÇC][ÃA]O\s+ARPE\s+N[ºO°]?\s*(\d+).*?DE\s+(\d{1,2})\s+DE\s+(\w+)\s+DE\s+(\d{4})",
    re.IGNORECASE,
)

## Etapa 1 — Buscar a listagem

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
resp = httpx.get(URL_RESOLUCOES, headers=headers, timeout=30, follow_redirects=True)
print(f"HTTP {resp.status_code} — {len(resp.text)} chars")

soup = BeautifulSoup(resp.text, "lxml")
intro = soup.find("div", class_="custom-article-introtext")
print("bloco de conteúdo encontrado:", intro is not None)

## Etapa 2 — Parear título/data (texto solto) com o link `.pdf` seguinte,
por ordem de aparição no HTML (varredura linear dos descendentes)

In [ ]:
def listar_arpe_resolucoes(intro_soup, url_base: str) -> list[dict]:
    itens, vistos = [], set()
    titulo_atual = data_atual = numero_atual = None

    for node in intro_soup.descendants:
        if isinstance(node, NavigableString):
            texto = str(node).strip()
            if not texto:
                continue
            m = TITULO_RE.search(texto)
            if m:
                numero_atual, dia, mes_nome, ano = m.groups()
                mes = MESES_PT.get(mes_nome.lower())
                data_atual = f"{ano}-{mes:02d}-{int(dia):02d}" if mes else None
                titulo_atual = texto
        elif getattr(node, "name", None) == "a":
            href = node.get("href", "").strip()
            if not href.lower().endswith(".pdf"):
                continue
            url_abs = urllib.parse.urljoin(url_base, href)
            if url_abs in vistos:
                continue
            vistos.add(url_abs)
            itens.append({
                "numero": numero_atual,
                "data": data_atual,
                "titulo": titulo_atual,
                "url": url_abs,
            })
    return itens


itens = listar_arpe_resolucoes(intro, URL_RESOLUCOES)
print(f"Total de resoluções com PDF direto: {len(itens)}")

sem_data = [i for i in itens if not i["data"]]
sem_titulo = [i for i in itens if not i["titulo"]]
print(f"Sem data: {len(sem_data)}  Sem título: {len(sem_titulo)}")

## Etapa 3 — Amostra pra conferência manual (mais recentes e mais antigas)

In [ ]:
print("--- 5 mais recentes ---")
for i in itens[:5]:
    print(i["numero"], i["data"], "|", (i["titulo"] or "")[:90])
    print("   ", i["url"])

print("\n--- 5 mais antigas ---")
for i in itens[-5:]:
    print(i["numero"], i["data"], "|", (i["titulo"] or "")[:90])
    print("   ", i["url"])

## Etapa 4 — Quantificar as resoluções sem PDF direto (só página de detalhe)

Fica de fora do escopo desta integração — documentado como pendência
conhecida.

In [ ]:
todas_ocorrencias = TITULO_RE.findall(resp.text)
numeros_todos = {t[0] for t in todas_ocorrencias}
numeros_com_pdf = {i["numero"] for i in itens}
sem_pdf = sorted(numeros_todos - numeros_com_pdf, key=int, reverse=True)

print(f"Resoluções mencionadas na listagem: {len(numeros_todos)}")
print(f"Com PDF direto (capturáveis pelo dispatcher genérico): {len(numeros_com_pdf)}")
print(f"Sem PDF direto (só página HTML de detalhe, fora de escopo por ora): {len(sem_pdf)}")
print(sem_pdf)

## Conclusão da Fase 1

Confirmar antes de prosseguir pra Fase 2: (a) `sem_data`/`sem_titulo` vazios;
(b) amostra de título/data corresponde ao que se vê na página real (conferir
manualmente no navegador); (c) URLs de PDF resolvem (baixáveis).

Se tudo confirmado: **encaixa no dispatcher genérico `ingest-PDF`**, com uma
função `listar_arpe_resolucoes()` própria (mesmo padrão já usado pro CCEE —
`listar` customizado por fonte em `CONFIGS_FONTES`), cobrindo as ~300
resoluções com PDF direto. As ~25 sem PDF direto (texto só em página HTML)
ficam registradas como pendência conhecida em `controle_fontes.notas`, para
resolver depois (exigiria extrair texto de página HTML em vez de PDF — fora
do contrato atual de `processar_pdf()`).